 ##### Step 1: First I am loading all the source tables

In [1]:
from pyspark.sql.functions import *

df_customers = spark.read.table("bronze_olist_customers_dataset")
df_order_items = spark.read.table("bronze_olist_order_items_dataset")
df_order_payments = spark.read.table("bronze_olist_order_payments_dataset")
df_orders = spark.read.table("bronze_olist_orders_dataset")
df_products = spark.read.table("bronze_olist_products_dataset")


StatementMeta(, cbc945ed-3f3c-4b7f-97d4-7818f25b7f89, 3, Finished, Available, Finished, False)

In [ ]:
# Checking if all tables are loaded in dataframe correctly
# display(df_customers)
# display(df_order_items)
# display(df_order_payments)
# display(df_orders)
# display(df_products)

##### Step 2: Handling customers without customer_id (Basically null customer_id)

In [2]:
df_customers_clean = df_customers.filter("customer_id is not null")

#Note: We can use any other way also if business users need customer 
#records without id also for that we can then predecide and keep some id for such users

StatementMeta(, cbc945ed-3f3c-4b7f-97d4-7818f25b7f89, 4, Finished, Available, Finished, False)

##### Step 3: Casting columns as per requirement for order_items

In [3]:
from pyspark.sql.functions import col 
df_order_items_clean = df_order_items.withColumn('shipping_limit_date',col('shipping_limit_date').cast('date'))\
.withColumn('price',col('price').cast('float'))\
.withColumn('freight_value',col('freight_value').cast('float'))

StatementMeta(, cbc945ed-3f3c-4b7f-97d4-7818f25b7f89, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 06ce693d-8112-4119-92f3-04e0719966c9)

##### Step 4: Handling negative and null payment values and casting as per need

In [4]:
from pyspark.sql.functions import col

df_order_payments_clean = df_order_payments.filter(
    (col("payment_value") >= 0) & (col("payment_value").isNotNull())
).withColumn('payment_installments',col('payment_installments').cast('int'))\
.withColumn('payment_value',col('payment_value').cast('float'))

StatementMeta(, cbc945ed-3f3c-4b7f-97d4-7818f25b7f89, 6, Finished, Available, Finished, False)

##### Step 5: Changing date columns for orders as per required format

In [5]:
from pyspark.sql.functions import to_timestamp
df_orders_clean = df_orders\
.withColumn('order_purchase_timestamp',to_timestamp(col('order_purchase_timestamp'),'yyyy-MM-dd HH:mm:ss'))\
.withColumn('order_approved_at',to_timestamp(col('order_approved_at'),'yyyy-MM-dd HH:mm:ss'))\
.withColumn('order_delivered_carrier_date',to_timestamp(col('order_delivered_carrier_date'),'yyyy-MM-dd HH:mm:ss'))\
.withColumn('order_delivered_customer_date',to_timestamp(col('order_delivered_customer_date'),'yyyy-MM-dd HH:mm:ss'))\
.withColumn('order_estimated_delivery_date',to_timestamp(col('order_estimated_delivery_date'),'yyyy-MM-dd HH:mm:ss'))\
.dropDuplicates(["order_id"])

StatementMeta(, cbc945ed-3f3c-4b7f-97d4-7818f25b7f89, 7, Finished, Available, Finished, False)

##### Step 6: For products casting of numeric columns is done

In [6]:
df_products_clean = df_products\
.withColumn('product_name_lenght',col('product_name_lenght').cast('int'))\
.withColumn('product_description_lenght',col('product_description_lenght').cast('int'))\
.withColumn('product_photos_qty',col('product_photos_qty').cast('int'))\
.withColumn('product_weight_g',col('product_weight_g').cast('float'))\
.withColumn('product_length_cm',col('product_length_cm').cast('float'))\
.withColumn('product_height_cm',col('product_height_cm').cast('float'))\
.withColumn('product_width_cm',col('product_width_cm').cast('float'))


StatementMeta(, cbc945ed-3f3c-4b7f-97d4-7818f25b7f89, 8, Finished, Available, Finished, False)

##### Step 7: Writing clean dataframes to Silver tables

In [7]:
df_customers_clean.write.mode("overwrite").saveAsTable("silver_customers")
df_order_items_clean.write.mode("overwrite").saveAsTable("silver_order_items")
df_order_payments_clean.write.mode("overwrite").saveAsTable("silver_order_payments")
df_orders_clean.write.mode("overwrite").saveAsTable("silver_orders")
df_products_clean.write.mode("overwrite").saveAsTable("silver_products")


StatementMeta(, cbc945ed-3f3c-4b7f-97d4-7818f25b7f89, 9, Finished, Available, Finished, False)

##### Loading Silver tables in dataframes

In [8]:
df_customer_silver = spark.read.table("silver_customers")
df_order_items_silver = spark.read.table("silver_order_items")
df_order_payments_silver = spark.read.table("silver_order_payments")
df_orders_silver = spark.read.table("silver_orders")
df_products_silver = spark.read.table("silver_products")


StatementMeta(, cbc945ed-3f3c-4b7f-97d4-7818f25b7f89, 10, Finished, Available, Finished, False)

##### Aggregating payment values for single order_id

In [9]:
from pyspark.sql.functions import col,sum,round
df_payments_agg = df_order_payments_silver.groupBy(col("order_id"))\
.agg(round(sum(col("payment_value")),2).alias("total_payment_value"))

StatementMeta(, cbc945ed-3f3c-4b7f-97d4-7818f25b7f89, 11, Finished, Available, Finished, False)

##### Aggregating order items for single order_id

In [10]:
from pyspark.sql.functions import sum , count

df_items_agg = df_order_items_silver.groupBy("order_id") \
    .agg(
        sum("price").alias("total_item_price"),
        sum("freight_value").alias("total_freight"),
        count("order_item_id").alias("total_items")
    )

StatementMeta(, cbc945ed-3f3c-4b7f-97d4-7818f25b7f89, 12, Finished, Available, Finished, False)

##### Joining datasets (customers + orders+ payments + order_items)

In [11]:
df_joined = df_orders_silver.alias('o').join\
(df_customer_silver.alias('c'),"customer_id",'inner')
#display(df_joined)

StatementMeta(, cbc945ed-3f3c-4b7f-97d4-7818f25b7f89, 13, Finished, Available, Finished, False)

In [12]:
df_joined = df_orders_silver.alias('o').join\
(df_customer_silver.alias('c'),"customer_id",'inner')
#display(df_joined)

StatementMeta(, cbc945ed-3f3c-4b7f-97d4-7818f25b7f89, 14, Finished, Available, Finished, False)

In [13]:
df_joined = df_joined.alias('j').join\
(df_payments_agg.alias('p'),"order_id",'left')
#display(df_joined)

StatementMeta(, cbc945ed-3f3c-4b7f-97d4-7818f25b7f89, 15, Finished, Available, Finished, False)

In [14]:
df_joined = df_joined.alias('j').join\
(df_items_agg.alias('oi'),"order_id",'left')
#display(df_joined)

StatementMeta(, cbc945ed-3f3c-4b7f-97d4-7818f25b7f89, 16, Finished, Available, Finished, False)

##### Adding Business Columns

In [15]:
df_joined = df_joined.withColumn(
    "order_value",round((col("total_item_price") + col("total_freight")),2)
)\
.withColumn(
    "is_delivered",(col("order_status")=='delivered').cast("int")
)
#display(df_joined)

StatementMeta(, cbc945ed-3f3c-4b7f-97d4-7818f25b7f89, 17, Finished, Available, Finished, False)

##### Handling nulls in value columns

In [16]:
df_joined =df_joined.fillna({
    "total_payment_value":0,
    "order_value":0
})

StatementMeta(, cbc945ed-3f3c-4b7f-97d4-7818f25b7f89, 18, Finished, Available, Finished, False)

##### Selecting only required columns from the df_joined

In [19]:
df_final = df_joined.select(
    "order_id",
    "customer_id",
    "order_status",
    "order_purchase_timestamp",
    "total_payment_value",
    "total_items",
    "order_value",
    "is_delivered"
)


StatementMeta(, cbc945ed-3f3c-4b7f-97d4-7818f25b7f89, 21, Finished, Available, Finished, False)

##### Finally writing to Silver table

In [20]:
df_final.write.mode("overwrite").saveAsTable("silver_orders_enriched")

StatementMeta(, cbc945ed-3f3c-4b7f-97d4-7818f25b7f89, 22, Finished, Available, Finished, False)

##### Validation check

In [18]:
display(df_final.groupBy("order_id").count().filter("count > 1"))

StatementMeta(, cbc945ed-3f3c-4b7f-97d4-7818f25b7f89, 20, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d8bd8fe5-c7be-4254-bb35-d828a03c64f6)

In [21]:
df_final.filter("order_id IS NULL").count()

StatementMeta(, cbc945ed-3f3c-4b7f-97d4-7818f25b7f89, 23, Finished, Available, Finished, False)

0

In [23]:
df_final.filter("total_payment_value < 0").count()

StatementMeta(, cbc945ed-3f3c-4b7f-97d4-7818f25b7f89, 25, Finished, Available, Finished, False)

0